# 短期记忆

## 1、基于内存的持久化器

### 1.1 举例1：没有记忆

In [1]:
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from typing import Union, Dict, Any
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langgraph.config import get_stream_writer
from rich import print as rprint
from utils.tool_utils import query_weather_from_web,query_news_from_web

# 1.初始化模型给中间件使用，不是agent
model = ChatOpenAI(
    # model="qwen3-vl:latest",
    # model="483025889/qwen3.5:9b",
    model="gemma4:12b",
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
)

agent = create_agent(model)

print("\n第1轮对话：")
resp1 = agent.invoke({"messages":[HumanMessage("你好，我叫小国")]})
for msg in resp1["messages"]:
    msg.pretty_print()

print("\n第2轮对话：")
resp1 = agent.invoke({"messages":[HumanMessage("我叫什么名字？")]})
for msg in resp1["messages"]:
    msg.pretty_print()


第1轮对话：
================================ Human Message =================================

你好，我叫小国
================================== Ai Message ==================================

你好，小国！很高兴认识你。

我是你的AI助手。请问今天有什么我可以帮到你的吗？无论是想聊天、问问题，还是需要我帮你处理一些任务，都可以随时告诉我。

第2轮对话：
================================ Human Message =================================

我叫什么名字？
================================== Ai Message ==================================

抱歉，作为一个人工智能，我没有权限访问您的个人信息，所以**我不知道您的名字**。

如果您愿意告诉我，我很乐意称呼您的名字！😊


### 1.2 举例2：拥有记忆,基于checkpointer

In [2]:
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from typing import Union, Dict, Any
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from rich import print as rprint
from utils.tool_utils import query_weather_from_web,query_news_from_web

checkpoint = InMemorySaver()

# 1.初始化模型给中间件使用，不是agent
model = ChatOpenAI(
    # model="qwen3-vl:latest",
    # model="483025889/qwen3.5:9b",
    model="gemma4:12b",
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
)

# 3、同一个thread_id共享记忆的，不同的id就不能够共享记忆
config = {"configurable":{"thread_id":"1"}}
agent = create_agent(model,checkpointer=checkpoint)

print("\n第1轮对话：")
resp1 = agent.invoke({"messages":[HumanMessage("你好，我叫小国")]},config=config)
for msg in resp1["messages"]:
    msg.pretty_print()

print("\n第2轮对话：")
resp1 = agent.invoke({"messages":[HumanMessage("我叫什么名字？")]},config=config)
for msg in resp1["messages"]:
    msg.pretty_print()


第1轮对话：
================================ Human Message =================================

你好，我叫小国
================================== Ai Message ==================================

你好，小国！很高兴认识你。

请问今天有什么我可以帮你的吗？无论是你想聊天、咨询问题，还是需要我帮你完成某些任务，尽管告诉我！

第2轮对话：
================================ Human Message =================================

你好，我叫小国
================================== Ai Message ==================================

你好，小国！很高兴认识你。

请问今天有什么我可以帮你的吗？无论是你想聊天、咨询问题，还是需要我帮你完成某些任务，尽管告诉我！
================================ Human Message =================================

我叫什么名字？
================================== Ai Message ==================================

你叫**小国**。
